# Domain Analysis Demo

This notebook demonstrates the new domain analysis capabilities for your research data.
It implements all the analyses from your comprehensive analysis plan.

## Available Analyses:

### 1. Publication Trends
- Publication count over time (histogram)
- Domain trends (stacked column chart)
- Robotics research focus

### 2. Model Size Analysis
- Parameter size evolution (scatter plot)
- Domain parameter comparisons
- Modality parameter analysis
- Architecture parameter analysis
- Parameter trends with categorical coloring (domain, architecture, organization)

### 3. Modality Analysis
- Input/output modality trends
- Domain-modality matrices

### 4. Architecture Analysis
- Architecture trends over time
- Domain-architecture matrices


## Setup and Imports

### Run ID Range Configuration

You can set specific run ID ranges here to analyze only a subset of your data. 
This is useful for:
- Analyzing specific time periods
- Focusing on particular datasets  
- Comparing different extraction runs
- Debugging specific data ranges

**To analyze all data**: Leave both values as `None`
**To analyze specific range**: Set `RUN_ID_START` and `RUN_ID_END` to your desired range
**To analyze single run**: Set both to the same value


### Quick Run ID Range Examples

Here are some common run ID range configurations you can use:

```python
# Example configurations (uncomment one to use):

# 1. Analyze all data (default)
# RUN_ID_START = None
# RUN_ID_END = None

# 2. Analyze only recent data (last 2000 runs)
# RUN_ID_START = run_info['max_run_id'] - 2000
# RUN_ID_END = run_info['max_run_id']

# 3. Analyze only early data (first 2000 runs)
# RUN_ID_START = 1
# RUN_ID_END = 2000

# 4. Analyze specific range (e.g., runs 3000-5000)
# RUN_ID_START = 3000
# RUN_ID_END = 5000

# 5. Analyze single run (e.g., run 5000)
# RUN_ID_START = 5000
# RUN_ID_END = 5000

# 6. Analyze robotics-specific range (default robotics data)
# RUN_ID_START = 7866
# RUN_ID_END = 8087
```

**Note**: After changing the `RUN_ID_START` and `RUN_ID_END` values, re-run the setup cell above to see the updated configuration.


In [ ]:
# Import the new domain analysis capabilities
from analyser.domain_analysis import (
    DomainAnalyzer, 
    create_domain_modality_matrix, 
    create_domain_architecture_matrix
)
from analyser.trend_analysis import TrendAnalyzer, get_run_id_ranges
from analyser.config import get_config

# =============================================================================
# RUN ID RANGE CONFIGURATION
# =============================================================================
# Set these values to filter analysis to specific extraction run ID ranges
# None = analyze all available data
RUN_ID_START = 9976  # Starting run ID (inclusive)
RUN_ID_END = 10454    # Ending run ID (inclusive)



# Get available run ID information
run_info = get_run_id_ranges()
print(" Available Run ID Information:")
print(f"  • Run ID range: {run_info['min_run_id']} - {run_info['max_run_id']}")
print(f"  • Total runs: {run_info['total_runs']:,}")
print(f"  • Date range: {run_info['earliest_paper']} to {run_info['latest_paper']}")

# Display current configuration
if RUN_ID_START is None and RUN_ID_END is None:
    print(f"\n Analysis Configuration: ALL DATA (runs {run_info['min_run_id']}-{run_info['max_run_id']})")
elif RUN_ID_START == RUN_ID_END:
    print(f"\n Analysis Configuration: SINGLE RUN (run {RUN_ID_START})")
else:
    print(f"\n Analysis Configuration: RANGE (runs {RUN_ID_START}-{RUN_ID_END})")

# Initialize analyzers
domain_analyzer = DomainAnalyzer()
trend_analyzer = TrendAnalyzer()

print("\n Domain analysis tools loaded successfully!")
print(f" Available fields: {len(get_config()['common_fields'])} common fields")


In [ ]:
# =============================================================================
# ROBOTICS RANGE CONFIGURATION
# =============================================================================
# Set these values to use a specific range for robotics analysis only
# None = use the global RUN_ID_START and RUN_ID_END settings
ROBOTICS_RUN_ID_START = 7866  # Starting run ID for robotics analysis (inclusive)
ROBOTICS_RUN_ID_END = 8087    # Ending run ID for robotics analysis (inclusive)

# Determine robotics range
if ROBOTICS_RUN_ID_START is not None and ROBOTICS_RUN_ID_END is not None:
    # Use dedicated robotics range
    robotics_start = ROBOTICS_RUN_ID_START
    robotics_end = ROBOTICS_RUN_ID_END
    print(f" Robotics Analysis Setup (Using Dedicated Robotics Range)")
elif RUN_ID_START is not None and RUN_ID_END is not None:
    # Use global configured range
    robotics_start = RUN_ID_START
    robotics_end = RUN_ID_END
    print(f" Robotics Analysis Setup (Using Global Configured Range)")
else:
    # Use default robotics range
    robotics_start = 7866
    robotics_end = 8087
    print(f" Robotics Analysis Setup (Using Default Robotics Range)")

print(f" Robotics data range: {robotics_start} - {robotics_end}")
print(f" Total robotics runs: {robotics_end - robotics_start + 1}")
print("=" * 50)

## 1. Publication Trends Analysis


### 1a) Publication Count Over Time
**Insight**: Development speed over time (Accelerating, etc.)


In [ ]:
# Analyze publication trends over time
pub_trends = domain_analyzer.get_publication_trends(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    group_by='year',
    figsize=(14, 8)
)

print(f" Found {len(pub_trends)} years of publication data")
if not pub_trends.empty:
    print(f" Date range: {pub_trends['period'].min().year} - {pub_trends['period'].max().year}")
    print(f" Total publications: {pub_trends['publication_count'].sum():,}")


In [ ]:
df_years = domain_analyzer.get_papers_vs_models_by_year(
    run_id_start=robotics_start, 
    run_id_end=robotics_end,
    figsize=(16, 8)
)

### 1b) Domain Trends (Stacked Column Chart)
**Insight**: Which domains were in focus of research per year?


In [ ]:
# Analyze domain trends over time
domain_trends = domain_analyzer.get_domain_trends(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    top_k=10,
    figsize=(16, 10),
    
)

print(f"️ Analyzed top 10 domains over time")
if not domain_trends.empty:
    print(f" Total domains in dataset: {domain_trends.columns.nunique()}")


### 1c) Robotics Research Focus
**Insight**: When did the development of robotics focused models start? How is it developing?


In [ ]:
# Analyze robotics-specific trends
robotics_trends = domain_analyzer.get_robotics_trends(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    figsize=(14, 10)
)

print(f" Robotics research analysis completed")
if not robotics_trends.empty:
    print(f" Robotics research span: {robotics_trends['year'].min().year} - {robotics_trends['year'].max().year}")
    print(f" Total robotics publications: {robotics_trends['robotics_publications'].sum():,}")


## 2. Model Size Analysis


### 2a) Parameter Size Evolution
**Insight**: How is the parameter size evolving in general


In [ ]:
# Analyze parameter size trends over time
param_trends = domain_analyzer.get_parameter_size_analysis(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    log_scale=True,
    figsize=(14, 8)
)

print(f" Parameter size analysis completed")
if param_trends is not None and not param_trends.empty:
    numeric_count = param_trends['numeric_value'].notna().sum()
    print(f" Models with numeric parameters: {numeric_count:,}")
    if numeric_count > 0:
        max_params = param_trends['numeric_value'].max()
        min_params = param_trends['numeric_value'].min()
        print(f" Parameter range: {min_params:,.0f} - {max_params:,.0f}")


### 2b) Domain Parameter Comparison
**Insight**: How does parameter size of e.g. robotics compare to other domains?


In [ ]:
# Compare parameter sizes across domains
domain_param_comparison = domain_analyzer.get_domain_parameter_comparison(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    top_domains=10,
    figsize=(16, 10)
)

print(f" Domain parameter comparison completed")
if not domain_param_comparison.empty:
    print(f"️ Domains analyzed: {domain_param_comparison['domain'].nunique()}")
    print(f" Total models: {len(domain_param_comparison):,}")


### 2c) Modality Parameter Analysis
**Insight**: How do input/output modalities influence the parameter size?


In [ ]:
# Analyze parameter sizes by input modality
input_modality_trends = domain_analyzer.get_modality_analysis(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    modality_type='input',
    figsize=(14, 8)
)

print(f" Input modality analysis completed")
if not input_modality_trends.empty:
    print(f" Input modalities: {len(input_modality_trends.columns)}")


In [ ]:
# Analyze parameter sizes by output modality
output_modality_trends = domain_analyzer.get_modality_analysis(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    modality_type='output',
    figsize=(14, 8)
)

print(f" Output modality analysis completed")
if not output_modality_trends.empty:
    print(f" Output modalities: {len(output_modality_trends.columns)}")


### 2d) Architecture Parameter Analysis
**Insight**: How does the architecture influence the parameter size?


In [ ]:
# Analyze parameter sizes by architecture (excluding "Other" category)
architecture_trends = domain_analyzer.get_architecture_trends(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    top_k=10,
    figsize=(14, 8)
)

print(f"️ Architecture analysis completed")
if not architecture_trends.empty:
    print(f" Architectures analyzed: {len(architecture_trends.columns)}")


### 2e) Parameter Trends with Categorical Coloring
**Insight**: How do parameter trends vary across different categories (domains, architectures, etc.)? This analysis shows parameter evolution over time with points colored and shaped by categorical fields.


In [ ]:
# Analyze parameter trends colored by domain
param_trends_domain = domain_analyzer.get_parameter_trends_with_categories(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    field_name='parameters',
    color_by='domain',
    shape_by=None,
    top_k_categories=20,
    log_scale=True,
    figsize=(18, 12)
)

print(f" Parameter trends by domain analysis completed")
if not param_trends_domain.empty:
    print(f" Data points analyzed: {len(param_trends_domain):,}")
    print(f"️ Domains shown: {param_trends_domain['domain'].nunique()}")


In [ ]:
# Analyze parameter trends with both domain (color) and architecture (shape)
param_trends_multi = domain_analyzer.get_parameter_trends_with_categories(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    field_name='parameters',
    color_by='architecture',
    shape_by=None,
    top_k_categories=16,  # Fewer categories to avoid clutter
    log_scale=True,
    figsize=(18, 12),
)

print(f" Multi-category parameter trends analysis completed")
if not param_trends_multi.empty:
    #print(f" Data points analyzed: {len(param_trends_multi):,}")
    #print(f"️ Domains shown: {param_trends_multi['domain'].nunique()}")
    #print(f"️ Architectures shown: {param_trends_multi['architecture'].nunique()}")
    pass


In [ ]:
# Analyze parameter trends colored by organization
param_trends_org = domain_analyzer.get_parameter_trends_with_categories(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    field_name='parameters',
    color_by='organization',
    shape_by=None,
    top_k_categories=20,
    log_scale=True,
    figsize=(18, 12)
)

print(f" Parameter trends by organization analysis completed")
if not param_trends_org.empty:
    print(f" Data points analyzed: {len(param_trends_org):,}")
    print(f" Organizations shown: {param_trends_org['organization'].nunique()}")


In [ ]:
# Analyze parameter trends by architecture with spotlight effect
spotlight_data = domain_analyzer.get_parameter_trends_spotlight(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    field_name='parameters',
    category_field='architecture',
    top_k_categories=12,
    log_scale=True,
    figsize=(20, 32),
    largest_model_only=True,  # Default
    show_trend_lines=True  # Default
    
)

## 3. Modality Analysis


### 3a) Modality Trends Over Time
**Insight**: What modalities were processable over time?


In [ ]:
# Analyze input modality trends
input_modality_trends = domain_analyzer.get_modality_analysis(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    modality_type='input',
    figsize=(16, 10)
)

print(f" Input modality trends analyzed")
if not input_modality_trends.empty:
    print(f" Input modalities: {len(input_modality_trends.columns)}")
    print(f" Time span: {input_modality_trends.index.min().year} - {input_modality_trends.index.max().year}")


In [ ]:
# Analyze output modality trends
output_modality_trends = domain_analyzer.get_modality_analysis(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    modality_type='output',
    figsize=(16, 10)
)

print(f" Output modality trends analyzed")
if not output_modality_trends.empty:
    print(f" Output modalities: {len(output_modality_trends.columns)}")
    print(f" Time span: {output_modality_trends.index.min().year} - {output_modality_trends.index.max().year}")


### 3b) Domain-Modality Matrices
**Insight**: What modalities are used for domain specific models?


In [ ]:
# Create domain-input modality matrix
domain_input_matrix = create_domain_modality_matrix(
    domain_analyzer,
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    modality_type='input', 
    figsize=(14, 10)
)

print(f" Domain-Input Modality matrix created")
if not domain_input_matrix.empty:
    print(f"️ Domains: {len(domain_input_matrix.index)}")
    print(f" Input modalities: {len(domain_input_matrix.columns)}")


In [ ]:
# Create domain-output modality matrix
domain_output_matrix = create_domain_modality_matrix(
    domain_analyzer,
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    modality_type='output', 
    figsize=(14, 10)
)

print(f" Domain-Output Modality matrix created")
if not domain_output_matrix.empty:
    print(f"️ Domains: {len(domain_output_matrix.index)}")
    print(f" Output modalities: {len(domain_output_matrix.columns)}")


## 4. Architecture Analysis


### 4a) Architecture Trends Over Time
**Insight**: How has architecture usage changed over time?


## 📊 New Visualization Features

This notebook now demonstrates **both visualization types** for trend analysis:

### 🔄 **Visualization Type Options:**

1. **Default Visualizations** (Original):
   - **Grouped/Histogram Charts**: Show absolute counts for easy comparison
   - **Pie Charts**: Show distribution for categorical data

2. **New Stacked Percentage Charts** (Enhanced):
   - **Temporal Trends**: Show how percentages change over time
   - **Better Trend Understanding**: Reveals market share evolution
   - **Relative Comparisons**: Focus on proportional changes rather than absolute numbers

### 🎯 **When to Use Each:**
- **Histogram/Pie**: When you want to see absolute numbers and simple distributions
- **Stacked Percentage**: When you want to understand trends and how categories evolve over time

### 📋 **Functions with Dual Visualizations:**
- `get_architecture_trends()` - Architecture usage over time
- `get_robot_type_analysis()` - Robot types in robotics models
- `get_organization_analysis()` - Organization contributions over time
- `get_training_dataset_analysis()` - Dataset popularity trends
- `get_control_type_analysis()` - Control types in robotics
- `get_environment_type_analysis()` - Environment types in robotics

Each function now shows **both visualizations** for comprehensive analysis!


In [ ]:
# Analyze architecture trends over time
architecture_trends = domain_analyzer.get_architecture_trends(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    top_k=10,
    figsize=(16, 10)
)

print(f"️ Architecture trends analyzed")
if not architecture_trends.empty:
    print(f" Top architectures: {len(architecture_trends.columns)}")
    print(f" Time span: {architecture_trends.index.min().year} - {architecture_trends.index.max().year}")


In [ ]:
# Analyze architecture trends over time (Default Grouped Bars)
print(" Architecture Trends - Grouped Bar Visualization")
architecture_trends_grouped = domain_analyzer.get_architecture_trends(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    top_k=10,
    visualization_type='grouped',  # Default - grouped bars
    figsize=(16, 10)
)

print(f"️ Architecture trends analyzed (Grouped)")
if not architecture_trends_grouped.empty:
    print(f" Top architectures: {len(architecture_trends_grouped.columns)}")
    print(f" Time span: {architecture_trends_grouped.index.min().year} - {architecture_trends_grouped.index.max().year}")
else:
    print("️ No architecture data found")


In [ ]:
# Analyze architecture trends over time (Stacked Percentage Bars)
print(" Architecture Trends - Stacked Percentage Visualization")
architecture_trends_stacked = domain_analyzer.get_architecture_trends(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    top_k=10,
    visualization_type='stacked_percentage',  # New - stacked percentage bars
    figsize=(16, 10)
)

print(f"️ Architecture trends analyzed (Stacked Percentage)")
if not architecture_trends_stacked.empty:
    print(f" Top architectures: {len(architecture_trends_stacked.columns)}")
    print(f" Time span: {architecture_trends_stacked.index.min().year} - {architecture_trends_stacked.index.max().year}")
else:
    print("️ No architecture data found")


### 4b) Domain-Architecture Matrix
**Insight**: Are there preferred architectures in certain domains?


In [ ]:
# Create domain-architecture matrix
domain_arch_matrix = create_domain_architecture_matrix(
    domain_analyzer,
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    figsize=(14, 10)
)

print(f" Domain-Architecture matrix created")
if not domain_arch_matrix.empty:
    print(f"️ Domains: {len(domain_arch_matrix.index)}")
    print(f"️ Architectures: {len(domain_arch_matrix.columns)}")


## Summary and Next Steps


In [ ]:
print(" Domain Analysis Demo Completed!")
print("=" * 50)

print("\n Analyses Performed:")
print("   Publication trends over time")
print("   Domain focus analysis")
print("   Robotics research trends")
print("   Parameter size evolution")
print("   Domain parameter comparisons")
print("   Modality trend analysis")
print("   Architecture trend analysis")
print("   Cross-domain relationship matrices")

print("\n Key Insights Generated:")
print("  • Publication acceleration patterns")
print("  • Domain research focus shifts")
print("  • Robotics development timeline")
print("  • Parameter size scaling trends")
print("  • Modality adoption patterns")
print("  • Architecture preference evolution")

print("\n Next Steps:")
print("  1. Review the generated visualizations")
print("  2. Export data for further analysis")
print("  3. Create custom analysis scripts")
print("  4. Integrate with external datasets")
print("  5. Generate publication-ready figures")

print("\n Available Scripts:")
print("  • examples/comprehensive_domain_analysis.py - Full analysis suite")
print("  • examples/quick_insights.py - Quick insights only")
print("  • analyser/domain_analysis.py - Core analysis functions")


## Bonus: Data Exploration and Customization


### Explore Available Data


In [ ]:
# Explore what data is available
from analyser.trend_analysis import explore_available_fields, get_run_id_ranges

# Get run ID information
run_info = get_run_id_ranges()
print(" Database Information:")
print(f"  • Run ID range: {run_info['min_run_id']} - {run_info['max_run_id']}")
print(f"  • Total runs: {run_info['total_runs']:,}")
print(f"  • Unique papers: {run_info['unique_papers']:,}")
print(f"  • Date range: {run_info['earliest_paper']} to {run_info['latest_paper']}")

# Get available fields
fields = explore_available_fields()
print(f"\n Available Fields ({len(fields)} total):")
for i, (_, row) in enumerate(fields.head(15).iterrows(), 1):
    print(f"  {i:2d}. {row['field_name']:<25} | Count: {row['count']:,}")
if len(fields) > 15:
    print(f"  ... and {len(fields) - 15} more fields")


### Custom Analysis Examples


In [ ]:
# Example: Analyze specific time period
print(" Analyzing configured range...")

# Use the configured run ID range for analysis
recent_trends = domain_analyzer.get_publication_trends(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    group_by='year',
    figsize=(12, 6)
)

if not recent_trends.empty:
    print(f" Publications in configured range: {recent_trends['publication_count'].sum():,}")
    if len(recent_trends) > 1:
        print(f" Growth rate: {((recent_trends['publication_count'].iloc[-1] / recent_trends['publication_count'].iloc[0]) - 1) * 100:.1f}%")
else:
    print("️ No data found in configured range")


### Robotics Range Configuration

You can set a specific range for robotics analysis here. This will override the global `RUN_ID_START` and `RUN_ID_END` settings for robotics-specific functions.

**Options:**
- **Use global range**: Leave both as `None` (will use `RUN_ID_START` and `RUN_ID_END` from above)
- **Use default robotics range**: Leave both as `None` and set global range to `None` (uses runs 7866-8087)
- **Custom robotics range**: Set specific values for robotics analysis only

**Example configurations:**
```python
# 1. Use global range (default)
# ROBOTICS_RUN_ID_START = None
# ROBOTICS_RUN_ID_END = None

# 2. Analyze recent robotics data (runs 8000-8500)
# ROBOTICS_RUN_ID_START = 8000
# ROBOTICS_RUN_ID_END = 8500

# 3. Analyze early robotics data (runs 7000-7500)
# ROBOTICS_RUN_ID_START = 7000
# ROBOTICS_RUN_ID_END = 7500

# 4. Analyze single robotics run
# ROBOTICS_RUN_ID_START = 8000
# ROBOTICS_RUN_ID_END = 8000

# 5. Use default robotics range (runs 7866-8087)
# ROBOTICS_RUN_ID_START = 7866
# ROBOTICS_RUN_ID_END = 8087
```


In [ ]:
# Example: Export data for further analysis
print(" Exporting analysis results...")

# Save results to CSV files
if 'pub_trends' in locals() and not pub_trends.empty:
    #pub_trends.to_csv('publication_trends.csv', index=False)
    print(" Saved publication_trends.csv")

if 'domain_trends' in locals() and not domain_trends.empty:
    #domain_trends.to_csv('domain_trends.csv')
    print(" Saved domain_trends.csv")

if 'param_trends' in locals() and param_trends is not None and not param_trends.empty:
    #param_trends.to_csv('parameter_trends.csv', index=False)
    print(" Saved parameter_trends.csv")

print(" Data exported for further analysis!")


# 🤖 Robotics-Specific Analysis

This section demonstrates specialized analysis for robotics models (run IDs 7866-8087) with full array parsing support for multi-value fields.

## Available Robotics Analyses:

### 5. Robotics Applications
- **5a) Robot types histogram** (with temporal trends)
- **5b) Robotics modality analysis** (input & sensor modalities over time)
- **5c) Modality development** (number of modalities per model over time)
- **5d) Control type analysis** (pie chart)
- **5e) Environment type analysis** (pie chart)

**Array Parsing**: All methods automatically handle multi-value fields like `{manipulator, mobile}`, `[high_level, low_level]`, etc.


In [ ]:
# =============================================================================
# ROBOTICS RANGE CONFIGURATION
# =============================================================================
# Set these values to use a specific range for robotics analysis only
# None = use the global RUN_ID_START and RUN_ID_END settings
ROBOTICS_RUN_ID_START = 7866  # Starting run ID for robotics analysis (inclusive)
ROBOTICS_RUN_ID_END = 8087    # Ending run ID for robotics analysis (inclusive)

# Determine robotics range
if ROBOTICS_RUN_ID_START is not None and ROBOTICS_RUN_ID_END is not None:
    # Use dedicated robotics range
    robotics_start = ROBOTICS_RUN_ID_START
    robotics_end = ROBOTICS_RUN_ID_END
    print(f"🤖 Robotics Analysis Setup (Using Dedicated Robotics Range)")
elif RUN_ID_START is not None and RUN_ID_END is not None:
    # Use global configured range
    robotics_start = RUN_ID_START
    robotics_end = RUN_ID_END
    print(f"🤖 Robotics Analysis Setup (Using Global Configured Range)")
else:
    # Use default robotics range
    robotics_start = 7866
    robotics_end = 8087
    print(f"🤖 Robotics Analysis Setup (Using Default Robotics Range)")

print(f"📊 Robotics data range: {robotics_start} - {robotics_end}")
print(f"📊 Total robotics runs: {robotics_end - robotics_start + 1}")
print("=" * 50)


## 5a) Robot Type Analysis

Analyze robot types in robotics models with temporal trends.


In [ ]:
# Robot type analysis with temporal trends
robot_types = domain_analyzer.get_robot_type_analysis(
    run_id_start=robotics_start,
    run_id_end=robotics_end,
    include_temporal=True,
    figsize=(15, 8)
)

print(f" Robot type analysis completed")
if not robot_types.empty:
    print(f"️ Found {len(robot_types)} different robot types")
    print(f" Top 3 robot types:")
    for i, (_, row) in enumerate(robot_types.head(3).iterrows(), 1):
        print(f"  {i}. {row['robot_type']}: {row['count']} models")
else:
    print("️ No robot type data found")


In [ ]:
# Robot type analysis with temporal trends (Default Histogram)
print(" Robot Type Analysis - Histogram Visualization")
robot_types_histogram = domain_analyzer.get_robot_type_analysis(
    run_id_start=robotics_start,
    run_id_end=robotics_end,
    include_temporal=True,
    visualization_type='histogram',  # Default - histogram visualization
    figsize=(15, 8)
)

print(f" Robot type analysis completed (Histogram)")
if not robot_types_histogram.empty:
    print(f"️ Found {len(robot_types_histogram)} different robot types")
    print(f" Top 3 robot types:")
    for i, (_, row) in enumerate(robot_types_histogram.head(3).iterrows(), 1):
        print(f"  {i}. {row['robot_type']}: {row['count']} models")
else:
    print("️ No robot type data found")


In [ ]:
# Robot type analysis with temporal trends (Stacked Percentage)
print(" Robot Type Analysis - Stacked Percentage Visualization")
robot_types_stacked = domain_analyzer.get_robot_type_analysis(
    run_id_start=robotics_start,
    run_id_end=robotics_end,
    include_temporal=False,
    top_k=6,
    visualization_type='stacked_percentage',  # New - stacked percentage visualization
    figsize=(16, 7)
)

print(f" Robot type analysis completed (Stacked Percentage)")
if not robot_types_stacked.empty:
    print(f"️ Found {len(robot_types_stacked)} different robot types")
    print(f" Top 3 robot types:")
    for i, (_, row) in enumerate(robot_types_stacked.head(5).iterrows(), 1):
        print(f"  {i}. {row['robot_type']}: {row['count']} models")
else:
    print("️ No robot type data found")


## 5b) Robotics Modality Analysis

Analyze input and sensor modalities in robotics models over time.


In [ ]:
# Robotics modality analysis (input and sensor modalities over time)
robotics_modalities = domain_analyzer.get_robotics_modality_analysis(
    run_id_start=robotics_start,
    run_id_end=robotics_end,
    top_k=8,
    figsize=(15, 12)
)

print(f" Robotics modality analysis completed")
if not robotics_modalities.empty:
    input_modalities = robotics_modalities[robotics_modalities['modality_type'] == 'input']
    sensor_modalities = robotics_modalities[robotics_modalities['modality_type'] == 'sensor']
    
    print(f" Input modalities: {len(input_modalities['modality'].unique())} unique types")
    print(f" Sensor modalities: {len(sensor_modalities['modality'].unique())} unique types")
    
    if not input_modalities.empty:
        print(f" Top 3 input modalities:")
        top_input = input_modalities.groupby('modality')['count'].sum().nlargest(3)
        for i, (modality, count) in enumerate(top_input.items(), 1):
            print(f"  {i}. {modality}: {count} models")
    
    if not sensor_modalities.empty:
        print(f" Top 3 sensor modalities:")
        top_sensor = sensor_modalities.groupby('modality')['count'].sum().nlargest(3)
        for i, (modality, count) in enumerate(top_sensor.items(), 1):
            print(f"  {i}. {modality}: {count} models")
else:
    print("️ No robotics modality data found")


## 5c) Modality Development Analysis

Track how many modalities each model uses over time to see if models are becoming more capable.


In [ ]:
# Modality development analysis (number of modalities per model over time)
modality_development = domain_analyzer.get_modality_development_analysis(
    run_id_start=robotics_start,
    run_id_end=robotics_end,
    figsize=(14, 8)
)

print(f" Modality development analysis completed")
if not modality_development.empty:
    print(f" Analyzed {len(modality_development)} models")
    avg_modalities = modality_development['total_modality_count'].mean()
    print(f" Average modalities per model: {avg_modalities:.2f}")
    
    # Show year-by-year progression
    yearly_avg = modality_development.groupby('year')['total_modality_count'].mean()
    print(f" Modality progression by year:")
    for year, avg in yearly_avg.items():
        print(f"  {int(year)}: {avg:.2f} modalities per model")
    
    # Show distribution
    modality_dist = modality_development['total_modality_count'].value_counts().sort_index()
    print(f" Distribution of modality counts:")
    for count, freq in modality_dist.head(5).items():
        print(f"  {count} modalities: {freq} models")
else:
    print("️ No modality development data found")


In [ ]:
df_range1, df_range2 = domain_analyzer.compare_modality_development_ranges(
    range1_start=robotics_start, range1_end=robotics_end,  # First range
    range2_start=RUN_ID_START, range2_end=RUN_ID_END,  # Second range
    range1_label="Robotics",        # Custom label
    range2_label="EpochAI",        # Custom label
    normalize_by_count=True,              # Normalize by percentage
    figsize=(12, 6)                      # Large figure for 2x2 grid
)



## 5d) Control Type Analysis

Analyze control types to understand the depth of robot control implemented in models.


In [ ]:
# Control type analysis (Default Pie Chart)
print(" Control Type Analysis - Pie Chart Visualization")
control_types_pie = domain_analyzer.get_control_type_analysis(
    run_id_start=robotics_start,
    run_id_end=robotics_end,
    visualization_type='pie',  # Default - pie chart visualization
    figsize=(10, 8)
)

print(f" Control type analysis completed (Pie Chart)")
if not control_types_pie.empty:
    print(f"️ Found {len(control_types_pie)} different control types")
    print(f" Control type distribution:")
    for i, (_, row) in enumerate(control_types_pie.iterrows(), 1):
        print(f"  {i}. {row['control_type']}: {row['count']} models")
else:
    print("️ No control type data found")


In [ ]:
# Control type analysis (Stacked Percentage)
print(" Control Type Analysis - Stacked Percentage Visualization")
control_types_stacked = domain_analyzer.get_control_type_analysis(
    run_id_start=robotics_start,
    run_id_end=robotics_end,
    visualization_type='stacked_percentage',  # New - stacked percentage visualization
    figsize=(12, 8)
)

print(f" Control type analysis completed (Stacked Percentage)")
if not control_types_stacked.empty:
    print(f"️ Control types analyzed over time")
    print(f" Temporal analysis showing control type evolution")
    print(f" Control type trends displayed as percentages")
else:
    print("️ No temporal control type data found")


In [ ]:
print(" Control Type Analysis - Stacked Percentage Visualization")
control_types_stacked = domain_analyzer.get_control_type_spotlight(
    run_id_start=robotics_start,
    run_id_end=robotics_end,
    top_k_control_types=4,
    figsize=(24, 24)
)

print(f" Control type analysis completed (Stacked Percentage)")
if not control_types_stacked.empty:
    print(f"️ Control types analyzed over time")
    print(f" Temporal analysis showing control type evolution")
    print(f" Control type trends displayed as percentages")
else:
    print("️ No temporal control type data found")


In [ ]:
# Control type analysis (pie chart)
control_types = domain_analyzer.get_control_type_analysis(
    run_id_start=robotics_start,
    run_id_end=robotics_end,
    figsize=(12, 8)
)

print(f" Control type analysis completed")
if not control_types.empty:
    print(f"️ Found {len(control_types)} different control types")
    print(f" Top 3 control types:")
    for i, (_, row) in enumerate(control_types.head(3).iterrows(), 1):
        print(f"  {i}. {row['control_type']}: {row['count']} models")
    
    # Calculate percentages
    total_models = control_types['count'].sum()
    print(f" Control type distribution:")
    for _, row in control_types.head(5).iterrows():
        percentage = (row['count'] / total_models) * 100
        print(f"  {row['control_type']}: {row['count']} models ({percentage:.1f}%)")
else:
    print("️ No control type data found")


In [ ]:
matrix = domain_analyzer.get_robot_type_control_level_matrix(
    run_id_start=robotics_start,
    run_id_end=robotics_end,
    top_k_robot_types=6,
    show_percentages=True,
    figsize=(14, 10)
)

## 5e) Environment Type Analysis

Analyze environment types to see if models are developed for real-world conditions.


In [ ]:
# Environment type analysis (Default Pie Chart)
print(" Environment Type Analysis - Pie Chart Visualization")
environment_types_pie = domain_analyzer.get_environment_type_analysis(
    run_id_start=robotics_start,
    run_id_end=robotics_end,
    visualization_type='pie',  # Default - pie chart visualization
    figsize=(10, 8)
)

print(f" Environment type analysis completed (Pie Chart)")
if not environment_types_pie.empty:
    print(f"️ Found {len(environment_types_pie)} different environment types")
    print(f" Environment type distribution:")
    for i, (_, row) in enumerate(environment_types_pie.iterrows(), 1):
        print(f"  {i}. {row['environment_types']}: {row['count']} models")
else:
    print("️ No environment type data found")


In [ ]:
# Environment type analysis (Stacked Percentage)
print(" Environment Type Analysis - Stacked Percentage Visualization")
environment_types_stacked = domain_analyzer.get_environment_type_analysis(
    run_id_start=robotics_start,
    run_id_end=robotics_end,
    visualization_type='stacked_percentage',  # New - stacked percentage visualization
    figsize=(12, 8)
)

print(f" Environment type analysis completed (Stacked Percentage)")
if not environment_types_stacked.empty:
    print(f"️ Environment types analyzed over time")
    print(f" Temporal analysis showing environment type evolution")
    print(f" Environment type trends displayed as percentages")
else:
    print("️ No temporal environment type data found")


In [ ]:
# Environment type analysis (pie chart)
environment_types = domain_analyzer.get_environment_type_analysis(
    run_id_start=robotics_start,
    run_id_end=robotics_end,
    figsize=(12, 8)
)

print(f" Environment type analysis completed")
if not environment_types.empty:
    print(f"️ Found {len(environment_types)} different environment types")
    print(f" Top 3 environment types:")
    for i, (_, row) in enumerate(environment_types.head(3).iterrows(), 1):
        print(f"  {i}. {row['environment_types']}: {row['count']} models")
    
    # Calculate percentages
    total_models = environment_types['count'].sum()
    print(f" Environment type distribution:")
    for _, row in environment_types.head(5).iterrows():
        percentage = (row['count'] / total_models) * 100
        print(f"  {row['environment_types']}: {row['count']} models ({percentage:.1f}%)")
    
    # Check for real-world vs simulation
    real_world_keywords = ['real', 'physical', 'outdoor', 'indoor', 'household', 'lab']
    simulation_keywords = ['simulated', 'simulation', 'virtual']
    
    real_world_count = 0
    simulation_count = 0
    
    for _, row in environment_types.iterrows():
        env_type = row['environment_types'].lower()
        if any(keyword in env_type for keyword in real_world_keywords):
            real_world_count += row['count']
        if any(keyword in env_type for keyword in simulation_keywords):
            simulation_count += row['count']
    
    print(f"\n Environment focus:")
    print(f"  Real-world environments: {real_world_count} models ({real_world_count/total_models*100:.1f}%)")
    print(f"  Simulation environments: {simulation_count} models ({simulation_count/total_models*100:.1f}%)")
else:
    print("️ No environment type data found")


## 🤖 Robotics Analysis Summary

This section provided comprehensive robotics-specific analysis covering:

### ✅ **Completed Analyses:**
- **Robot Types**: Histogram and temporal trends showing which robot types are most commonly developed
- **Modalities**: Input and sensor modality trends over time in robotics models
- **Modality Development**: Tracking how many modalities each model uses (capability evolution)
- **Control Types**: Distribution of control approaches (high-level vs low-level)
- **Environment Types**: Analysis of real-world vs simulation environments

### 🔧 **Key Features:**
- **Array Parsing**: All methods handle multi-value fields like `{manipulator, mobile}`
- **Temporal Analysis**: Trends over time for robot types and modalities
- **Comprehensive Coverage**: All 5 robotics analysis areas from your research plan
- **Real-world Insights**: Environment analysis shows focus on practical applications

### 💡 **Research Questions Answered:**
- ✅ What robot types are most commonly developed for?
- ✅ What modalities are used in robotics models over time?
- ✅ Are models becoming more capable (more modalities)?
- ✅ What control depths are implemented?
- ✅ Are models developed for real-world conditions?


# 📊 Additional Analysis: Training Datasets & Organizations

This section provides analysis of training datasets and organizations involved in model development.


## 6a) Training Dataset Analysis

Analyze the most commonly used training datasets across all models.


In [ ]:
# Training dataset analysis
training_datasets = domain_analyzer.get_training_dataset_analysis(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    top_k=10,
    figsize=(15, 10),
    one_per_paper=True
)

print(f" Training dataset analysis completed")
if not training_datasets.empty:
    print(f"️ Found {len(training_datasets)} different training datasets")
    print(f" Top 10 training datasets:")
    for i, (_, row) in enumerate(training_datasets.head(10000).iterrows(), 1):
        print(f"  {i:2d}. {row['training_dataset']:<25}: {row['count']:>4} models")
    
    # Calculate total models
    total_models = training_datasets['count'].sum()
    print(f"\n Total models with training dataset info: {total_models:,}")
    
    # Show distribution insights
    top_5_percentage = (training_datasets.head(5)['count'].sum() / total_models) * 100
    print(f" Top 5 datasets represent {top_5_percentage:.1f}% of all models")
else:
    print("️ No training dataset data found")


In [ ]:
# Training dataset analysis (Default Histogram)
print(" Training Dataset Analysis - Histogram Visualization")
training_datasets_histogram = domain_analyzer.get_training_dataset_analysis(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    top_k=25,
    visualization_type='histogram',  # Default - histogram visualization
    figsize=(15, 10)
)

print(f" Training dataset analysis completed (Histogram)")
if not training_datasets_histogram.empty:
    print(f"️ Found {len(training_datasets_histogram)} different training datasets")
    print(f" Top 10 training datasets:")
    for i, (_, row) in enumerate(training_datasets_histogram.iterrows(), 1):
        print(f"  {i:2d}. {row['training_dataset']:<25}: {row['count']:>4} models")
    
    # Calculate total models
    total_models = training_datasets_histogram['count'].sum()
    print(f"\n Total models with training dataset info: {total_models:,}")
    
    # Show distribution insights
    top_5_percentage = (training_datasets_histogram.head(5)['count'].sum() / total_models) * 100
    print(f" Top 5 datasets represent {top_5_percentage:.1f}% of all models")
else:
    print("️ No training dataset data found")


In [ ]:
# Training dataset analysis (Stacked Percentage)
print(" Training Dataset Analysis - Stacked Percentage Visualization")
training_datasets_stacked = domain_analyzer.get_training_dataset_analysis(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    top_k=15,  # Reduced for better temporal visualization
    visualization_type='stacked_percentage',  # New - stacked percentage visualization
    figsize=(15, 10)
)

print(f" Training dataset analysis completed (Stacked Percentage)")
if not training_datasets_stacked.empty:
    print(f"️ Training datasets analyzed over time")
    print(f" Temporal analysis showing dataset popularity evolution")
    print(f" Top datasets' changing usage displayed as percentages")
else:
    print("️ No temporal training dataset data found")


In [ ]:
# Basic comparison (your hypothesis test)
results = domain_analyzer.get_dataset_specificity_analysis(
    target_domains=['Language', 'Vision'],
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END
)

# Temporal analysis to see evolution
temporal_results = domain_analyzer.get_dataset_specificity_temporal_analysis(
    target_domains=['language', 'vision'],
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END
)

# Compare more domains
extended_results = domain_analyzer.get_dataset_specificity_analysis(
    target_domains=['language', 'vision', 'audio', 'multimodal'],
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END
)

In [ ]:
# Basic domain analysis with N/A included
vision_datasets = domain_analyzer.get_training_dataset_analysis_by_domain(
    target_domain='Vision',
    include_na=True,
    top_k=15,
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    visualization_type='histogram',
    exclude_domains=['Language']
)

# Language domain analysis excluding N/A
language_datasets = domain_analyzer.get_training_dataset_analysis_by_domain(
    target_domain='Language', 
    include_na=True,
    visualization_type='histogram',
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    exclude_domains=['Vision', 'Audio', 'Multimodal']
)

In [ ]:
# Compare dataset transparency across tasks
tasks_to_compare = [
    (['language'], ['image'], 'Pure Language'),
    (['image'], ['language'], 'Pure Vision'),
    (['generation'], ['classification'], 'Generation'),
    (['classification'], ['generation'], 'Classification')
]

for include, exclude, label in tasks_to_compare:
    print(f"\n=== {label.upper()} TASKS ===")
    df = domain_analyzer.get_training_dataset_analysis_by_task(
        include_task_keywords=include,
        exclude_task_keywords=exclude,
        include_na=True,
        top_k=15,
        run_id_start=RUN_ID_START,
        run_id_end=RUN_ID_END,
        visualization_type='histogram',
        figsize=(15, 10)
    )

    print(f"\n{label} Task Analysis:")

## 6b) Organization Analysis

Analyze the organizations most involved in model development.


In [ ]:
# Organization analysis
organizations = domain_analyzer.get_organization_analysis(
    run_id_start=robotics_start,
    run_id_end=robotics_end,
    top_k=15,
    figsize=(16, 10),
    count_type='both', 
    sort_by='papers'  # or 'papers'
)

print(f" Organization analysis completed")
if not organizations.empty:
    print(f"️ Found {len(organizations)} different organizations")
    print(f" Top 10 organizations:")
    for i, (_, row) in enumerate(organizations.head(500).iterrows(), 1):
        print(f"  {i:2d}. {row['organization']:<25}: {row['count']:>4} models")
    
    # Calculate total models
    total_models = organizations['count'].sum()
    print(f"\n Total models with organization info: {total_models:,}")
    
    # Show distribution insights
    top_5_percentage = (organizations.head(5)['count'].sum() / total_models) * 100
    print(f" Top 5 organizations represent {top_5_percentage:.1f}% of all models")
    
    # Identify major players
    major_players = organizations[organizations['count'] >= 1000]
    if not major_players.empty:
        print(f"\n Major players (1000+ models):")
        for _, row in major_players.iterrows():
            print(f"  • {row['organization']}: {row['count']} models")
else:
    print("️ No organization data found")


In [ ]:
# Organization analysis (Default Histogram)
print(" Organization Analysis - Histogram Visualization")
organizations_histogram = domain_analyzer.get_organization_analysis(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    top_k=30,
    visualization_type='histogram',  # Default - histogram visualization
    figsize=(10, 10)
)

print(f" Organization analysis completed (Histogram)")
if not organizations_histogram.empty:
    print(f"️ Found {len(organizations_histogram)} different organizations")
    print(f" Top 10 organizations:")
    for i, (_, row) in enumerate(organizations_histogram.head(10).iterrows(), 1):
        print(f"  {i:2d}. {row['organization']:<25}: {row['count']:>4} models")
    
    # Calculate total models
    total_models = organizations_histogram['count'].sum()
    print(f"\n Total models with organization info: {total_models:,}")
    
    # Show distribution insights
    top_5_percentage = (organizations_histogram.head(5)['count'].sum() / total_models) * 100
    print(f" Top 5 organizations represent {top_5_percentage:.1f}% of all models")
else:
    print("️ No organization data found")


In [ ]:
# Organization analysis (Stacked Percentage)
print(" Organization Analysis - Stacked Percentage Visualization")
organizations_stacked = domain_analyzer.get_organization_analysis(
    run_id_start=RUN_ID_START,
    run_id_end=RUN_ID_END,
    top_k=10,  # Reduced for better temporal visualization
    visualization_type='stacked_percentage',  # New - stacked percentage visualization
    figsize=(15, 10)
)

print(f" Organization analysis completed (Stacked Percentage)")
if not organizations_stacked.empty:
    print(f"️ Organizations analyzed over time")
    print(f" Temporal analysis showing organization evolution")
    print(f" Top organizations' changing market share displayed as percentages")
else:
    print("️ No temporal organization data found")


In [ ]:
domain_analyzer.get_organization_type_analysis(
    run_id_start=RUN_ID_START, 
    run_id_end=RUN_ID_END, 
    visualization_type='stacked_percentage'
)

In [ ]:
domain_analyzer.get_organization_type_analysis(
    run_id_start=robotics_start, 
    run_id_end=robotics_end, 
    
    visualization_type='stacked_percentage'
)

## 📊 Training Dataset & Organization Analysis Summary

This section provided insights into:

### ✅ **Training Dataset Analysis:**
- **Dataset Popularity**: Most commonly used training datasets across all models
- **Array Parsing**: Handles multi-dataset entries like `{ImageNet, COCO, OpenImages}`
- **Distribution Insights**: Shows concentration of dataset usage

### ✅ **Organization Analysis:**
- **Institutional Contributions**: Organizations most involved in model development
- **Array Parsing**: Handles multi-organization entries like `{Google, OpenAI, Microsoft}`
- **Major Players**: Identifies organizations with significant model contributions

### 🔧 **Key Features:**
- **Horizontal Bar Charts**: Better readability for long names
- **Value Labels**: Precise counts on each bar
- **Configurable Top-K**: Focus on most relevant results
- **Array Support**: Correctly parses and counts multi-value entries
- **Statistical Insights**: Distribution analysis and major player identification

### 💡 **Research Insights:**
- Understanding which datasets are most influential in AI development
- Identifying key institutional players in AI research
- Analyzing concentration vs. diversity in dataset usage and organizational contributions


# 🌍 Geographical Distribution Analysis

This section demonstrates the new geographical analysis capabilities that map research organizations to countries and regions, showing how AI research is distributed globally over time.

## Available Geographical Analyses:

### 7. Country and Region Distribution
- **7a) Country distribution analysis** (stacked bar chart with percentages by year)
- **7b) Region distribution analysis** (stacked bar chart with percentages by year)

**Key Features**:
- **Data Cleaning**: Uses the same institution cleaning logic to properly split and clean organization names
- **Percentage Visualization**: Each year's bar totals to 100% for easy comparison
- **Organization Mapping**: Maps organizations to countries/regions using the `ai_orgs.csv` file
- **Automatic Filtering**: Excludes organizations with "n/a" mappings


## 7a) Country Distribution Analysis

Analyze the geographical distribution of research organizations by country over time. Each year's bar shows the percentage distribution, totaling 100%.


In [ ]:
# Country distribution analysis
country_distribution = domain_analyzer.get_country_distribution_analysis(
    run_id_start=robotics_start,  # Use configured range
    run_id_end=robotics_end,      # Use configured range
    top_k=10,                   # Show top 10 countries
    figsize=(16, 10)            # Large figure for better visibility
)

print(f" Country distribution analysis completed")
if not country_distribution.empty:
    print(f" Analyzed {len(country_distribution)} years")
    print(f"️ Top countries: {list(country_distribution.columns)}")
    
    # Show some statistics
    print(f"\n Country Statistics (Average % over time):")
    for country in country_distribution.columns:
        avg_percentage = country_distribution[country].mean()
        print(f"  • {country}: {avg_percentage:.1f}%")
    
    # Show year-by-year insights
    print(f"\n Year-by-year insights:")
    for year in country_distribution.index:
        year_data = country_distribution.loc[year]
        top_country = year_data.idxmax()
        top_percentage = year_data.max()
        print(f"  {year.year}: {top_country} leads with {top_percentage:.1f}%")
else:
    print("️ No country distribution data found")


## 7b) Region Distribution Analysis

Analyze the geographical distribution of research organizations by region over time. Each year's bar shows the percentage distribution, totaling 100%.


In [ ]:
# Region distribution analysis
region_distribution = domain_analyzer.get_region_distribution_analysis(
    run_id_start=robotics_start,  # Use configured range
    run_id_end=robotics_end,      # Use configured range
    figsize=(16, 10)            # Large figure for better visibility
)

print(f" Region distribution analysis completed")
if not region_distribution.empty:
    print(f" Analyzed {len(region_distribution)} years")
    print(f"️ Regions: {list(region_distribution.columns)}")
    
    # Show some statistics
    print(f"\n Region Statistics (Average % over time):")
    for region in region_distribution.columns:
        avg_percentage = region_distribution[region].mean()
        print(f"  • {region}: {avg_percentage:.1f}%")
    
    # Show year-by-year insights
    print(f"\n Year-by-year insights:")
    for year in region_distribution.index:
        year_data = region_distribution.loc[year]
        top_region = year_data.idxmax()
        top_percentage = year_data.max()
        print(f"  {year.year}: {top_region} leads with {top_percentage:.1f}%")
    
    # Calculate regional dominance
    print(f"\n Regional Analysis:")
    total_years = len(region_distribution)
    for region in region_distribution.columns:
        leading_years = (region_distribution[region] == region_distribution.max(axis=1)).sum()
        dominance_percentage = (leading_years / total_years) * 100
        print(f"  • {region}: Leading in {leading_years}/{total_years} years ({dominance_percentage:.1f}%)")
else:
    print("️ No region distribution data found")


## 🌍 Geographical Analysis Summary

This section provided comprehensive geographical analysis covering:

### ✅ **Completed Analyses:**
- **Country Distribution**: Stacked bar chart showing percentage distribution of countries by year
- **Region Distribution**: Stacked bar chart showing percentage distribution of regions by year
- **Temporal Trends**: Year-by-year analysis of geographical shifts in AI research
- **Dominance Analysis**: Identification of leading countries/regions over time

### 🔧 **Key Features:**
- **Data Cleaning**: Uses the same institution cleaning logic as the `clean_institutions.py` script
- **Organization Mapping**: Maps organizations to countries/regions using `ai_orgs.csv`
- **Percentage Visualization**: Each year totals to 100% for easy comparison
- **Automatic Filtering**: Excludes organizations with "n/a" mappings
- **Comprehensive Statistics**: Average percentages, year-by-year insights, and dominance analysis

### 💡 **Research Questions Answered:**
- ✅ Which countries are leading AI research over time?
- ✅ How has the geographical distribution of AI research evolved?
- ✅ Which regions dominate AI research globally?
- ✅ Are there geographical shifts in AI research focus?
- ✅ What is the relative contribution of different countries/regions?

### 📊 **Data Quality:**
- **Organization Mapping**: 613 organization mappings loaded from `ai_orgs.csv`
- **Data Cleaning**: Handles complex organization names with multiple separators
- **Array Parsing**: Correctly splits multi-organization entries
- **Quality Control**: Automatically filters out incomplete mappings


## 📊 Updated Analysis Summary

This comprehensive demo now includes all major analysis categories:

### ✅ **All Completed Analyses:**
- **Publication Trends**: Publication count over time, domain focus, robotics research
- **Model Size Analysis**: Parameter evolution, domain comparisons, modality/architecture analysis
- **Modality Analysis**: Input/output modality trends, domain-modality matrices
- **Architecture Analysis**: Architecture trends, domain-architecture matrices
- **Robotics-Specific Analysis**: Robot types, modalities, control types, environments
- **Training Dataset & Organization Analysis**: Dataset popularity, institutional contributions
- **🌍 Geographical Analysis**: Country and region distribution over time (NEW!)

### 🚀 **New Capabilities Added:**
- **Geographical Distribution**: Visualize how AI research is distributed globally
- **Country/Region Trends**: Track shifts in research leadership over time
- **Organization Mapping**: Automatic mapping of institutions to geographical locations
- **Percentage Visualization**: Easy comparison with 100% totals per year


In [ ]:
# Show current run ID configuration
print(" Current Run ID Configuration:")
if RUN_ID_START is None and RUN_ID_END is None:
    print(f"   Analyzing ALL DATA (runs {run_info['min_run_id']}-{run_info['max_run_id']})")
elif RUN_ID_START == RUN_ID_END:
    print(f"   Analyzing SINGLE RUN (run {RUN_ID_START})")
else:
    print(f"   Analyzing RANGE (runs {RUN_ID_START}-{RUN_ID_END})")

print("\n Run ID Filtering Tips:")
print("  • Set RUN_ID_START and RUN_ID_END to None to analyze all data")
print("  • Set both to the same value to analyze a single run")
print("  • Use get_run_id_ranges() to see available data ranges")
print("  • All analysis methods support run_id_start and run_id_end parameters")
print("  • Re-run the setup cell after changing RUN_ID_START and RUN_ID_END values")


In [ ]:
print(" Complete Domain Analysis Demo with Geographical Analysis!")
print("=" * 70)

print("\n All Analysis Categories Completed:")
print("   Publication trends over time")
print("   Domain focus analysis")
print("   Robotics research trends")
print("   Parameter size evolution")
print("   Domain parameter comparisons")
print("   Modality trend analysis")
print("   Architecture trend analysis")
print("   Cross-domain relationship matrices")
print("   Robotics-specific analysis (robot types, modalities, control, environments)")
print("   Training dataset and organization analysis")
print("    Country and region distribution analysis (NEW!)")

print("\n Key Insights Generated:")
print("  • Publication acceleration patterns")
print("  • Domain research focus shifts")
print("  • Robotics development timeline")
print("  • Parameter size scaling trends")
print("  • Modality adoption patterns")
print("  • Architecture preference evolution")
print("  • Training dataset popularity")
print("  • Institutional contributions")
print("   • Geographical distribution of AI research")
print("   • Country and region leadership trends")

print("\n Next Steps:")
print("  1. Review the generated visualizations")
print("  2. Export data for further analysis")
print("  3. Create custom analysis scripts")
print("  4. Integrate with external datasets")
print("  5. Generate publication-ready figures")
print("  6. Analyze geographical research patterns")

print("\n Available Scripts:")
print("  • examples/comprehensive_domain_analysis.py - Full analysis suite")
print("  • examples/quick_insights.py - Quick insights only")
print("  • examples/country_region_analysis_example.py - Geographical analysis example")
print("  • analyser/domain_analysis.py - Core analysis functions")
print("  • clean_institutions.py - Institution cleaning utility")
